# This notebook is used to do additional preprocessing after completing fmriprep

In [1]:
import pandas as pd
import numpy as np
import os.path as osp
import os

## SELECT PRJ FOLDER AND STUDY PARAMS

In [2]:
PRJ = 'preproc_test_bball'
task = 'view' #task is required
run = '01' #run is optional, if you do not want run just put ''

In [3]:
PRJ_DIR = f'/project/mdrosenberg/IG/{PRJ}'
fmriprep_DIR = osp.join(PRJ_DIR,'derivatives','fmriprep')
SBATCH_FILE = osp.join(PRJ_DIR,'code/SBATCH/S02_preproc.SBATCH.sh')
preprocessing_script = f'{PRJ_DIR}/code/bash/S02_preprocessing.sh'

In [5]:
def get_sub_list(DATA_DIR):
    import os

    # Get the list of files in the current directory
    files = os.listdir(DATA_DIR)
    
    # Sort the files by modification time
    sorted_files = sorted(files)
    
    sub_list = []

    for i, filename in enumerate(sorted_files):
        if filename[0:3] == 'sub':
            sub = filename
            sub_folder = osp.join(DATA_DIR, filename)
            if os.path.isdir(sub_folder):
                ses_count = 0
                ses_nums = []

                for sub_filename in os.listdir(sub_folder):
                    if sub_filename[0:3] == 'ses':
                        ses_count += 1
                        ses_nums.append(sub_filename)

                for s in range(ses_count):
                    sub_list.append((sub,ses_nums[s]))
    return sub_list

In [6]:
sub_list = sorted(get_sub_list(fmriprep_DIR))

Loop through sub_list and write file to run preprocessing

In [7]:
#note: ses is optional, if you do not want session just put ses='' in the command

to_run = 0

with open(SBATCH_FILE,'w') as the_file:
    the_file.write(f'#sh {SBATCH_FILE} \n')
    for sub in sub_list:
        fmriprep_file = osp.join(fmriprep_DIR, sub, 'func', f'{sub}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz')
        netcc_file = osp.join(PRJ_DIR, 'derivatives','preprocessed',sub, f'4_{sub}_task-{task}_LPI_000.netcc')
        if osp.exists(fmriprep_file):
            if not osp.exists(netcc_file):
                the_file.write(f'export sub={sub} dir={PRJ_DIR} task={task} fd_thresh={0.25} low_pass={0.01} high_pass={0.1}; sbatch {preprocessing_script} \n')
                to_run +=1
        elif not osp.exists(fmriprep_file):
            print(f'fmriprep issue for {sub}')
    the_file.close()

To execute this step:
1. Go to the SBATCH directory
2. ```bash
   sh S02_preprocessing.SBATCH.sh 
   ```

OR (from anywhere) run the command at the top of S02_preprocessing.SBATCH.sh

the .out and .err files will be written wherever you run the command from (they are numbered in order, so the first scan is the lowest number)